# Train BiLSTM and BiLSTM+Attention with Pandas Data

In [1]:
from __future__ import annotations

import argparse
from dataclasses import dataclass
from typing import Dict, Iterable, List, Sequence, Tuple

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

from attention import BiLSTMAttentionSentimentClassifier
from model import BiLSTMSentimentClassifier
from preprocessing import (
    DEFAULT_STOPWORDS,
    KoreanSentimentDataset,
    clean_text,
    keep_korean_only,
    pad_sequence,
    preprocess_dataframe,
    remove_stopwords,
    tokenize,
    tokens_to_ids,
)

DEFAULT_URL = 'https://raw.githubusercontent.com/skn29teacher/skn29_lecture/main/data_nlp/daum_movie_review.csv'

@dataclass
class TrainConfig:
    max_len: int = 30
    min_freq: int = 1
    batch_size: int = 32
    epochs: int = 3
    lr: float = 1e-3
    val_ratio: float = 0.2
    seed: int = 42
    embedding_dim: int = 64
    hidden_dim: int = 128
    attention_dim: int = 64
    dropout: float = 0.3
    rating_threshold: int = 6

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def load_raw_dataframe(csv_path: str | None, csv_url: str) -> pd.DataFrame:
    if csv_path:
        return pd.read_csv(csv_path)
    return pd.read_csv(csv_url)

def normalize_review_rating_columns(df: pd.DataFrame) -> pd.DataFrame:
    if 'review' in df.columns:
        review_col = 'review'
    else:
        review_col = df.columns[0]

    if 'rating(1~10점)' in df.columns:
        rating_col = 'rating(1~10점)'
    elif 'rating' in df.columns:
        rating_col = 'rating'
    else:
        rating_col = df.columns[1]

    out = df[[review_col, rating_col]].copy()
    out.columns = ['review', 'rating']
    return out

def build_train_dataframe(df: pd.DataFrame, threshold: int) -> pd.DataFrame:
    out = normalize_review_rating_columns(df)
    out['sentiment'] = (out['rating'] >= threshold).astype(int)
    return out[['review', 'sentiment']]

def prepare_dataset(
    train_df: pd.DataFrame,
    max_len: int,
    min_freq: int,
    stopwords: Iterable[str],
) -> Tuple[KoreanSentimentDataset, Dict[str, int], pd.DataFrame]:
    processed_df, vocab = preprocess_dataframe(
        df=train_df,
        text_col='review',
        label_col='sentiment',
        max_len=max_len,
        stopwords=stopwords,
        min_freq=min_freq,
    )
    dataset = KoreanSentimentDataset(processed_df)
    return dataset, vocab, processed_df

def split_loaders(
    dataset: KoreanSentimentDataset,
    batch_size: int,
    val_ratio: float,
    seed: int,
) -> Tuple[DataLoader, DataLoader]:
    val_size = max(1, int(len(dataset) * val_ratio)) if len(dataset) > 1 else 0
    train_size = len(dataset) - val_size

    if val_size > 0:
        train_dataset, val_dataset = random_split(
            dataset,
            [train_size, val_size],
            generator=torch.Generator().manual_seed(seed),
        )
    else:
        train_dataset = dataset
        val_dataset = dataset

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

def compute_accuracy(probs: torch.Tensor, labels: torch.Tensor) -> float:
    preds = (probs >= 0.5).float()
    return (preds == labels).float().mean().item()

def unwrap_probs(output):
    if isinstance(output, tuple):
        return output[0]
    return output

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    total_batches = 0

    for input_ids, labels, lengths in dataloader:
        input_ids = input_ids.to(device)
        labels = labels.to(device).float().unsqueeze(1)
        lengths = lengths.to(device)

        optimizer.zero_grad(set_to_none=True)
        output = model(input_ids, lengths)
        probs = unwrap_probs(output)
        loss = criterion(probs, labels)
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item())
        total_acc += compute_accuracy(probs.detach(), labels.detach())
        total_batches += 1

    avg_loss = total_loss / max(1, total_batches)
    avg_acc = total_acc / max(1, total_batches)
    return avg_loss, avg_acc

def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    total_batches = 0

    with torch.no_grad():
        for input_ids, labels, lengths in dataloader:
            input_ids = input_ids.to(device)
            labels = labels.to(device).float().unsqueeze(1)
            lengths = lengths.to(device)

            output = model(input_ids, lengths)
            probs = unwrap_probs(output)
            loss = criterion(probs, labels)

            total_loss += float(loss.item())
            total_acc += compute_accuracy(probs, labels)
            total_batches += 1

    avg_loss = total_loss / max(1, total_batches)
    avg_acc = total_acc / max(1, total_batches)
    return avg_loss, avg_acc

def fit_model(model, train_loader, val_loader, epochs, lr, device, name):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    history = []

    print(f'\n[{name}] start training')
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate_model(model, val_loader, criterion, device)
        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_loss,
            'val_acc': val_acc,
        })
        print(
            f'[{name}] epoch {epoch}: '
            f'train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, '
            f'val_loss={val_loss:.4f}, val_acc={val_acc:.4f}'
        )

    return history

def preprocess_texts_for_inference(texts, vocab, max_len):
    token_lists = []
    padded_ids = []
    lengths = []
    pad_idx = vocab['<pad>']

    for text in texts:
        processed = clean_text(text)
        processed = keep_korean_only(processed)
        tokens = tokenize(processed)
        tokens = remove_stopwords(tokens, stopwords=DEFAULT_STOPWORDS)
        ids = tokens_to_ids(tokens, vocab)
        length = max(1, min(len(ids), max_len))
        padded = pad_sequence(ids, max_len=max_len, pad_idx=pad_idx)

        token_lists.append(tokens)
        padded_ids.append(padded)
        lengths.append(length)

    input_ids = torch.tensor(padded_ids, dtype=torch.long)
    lengths_tensor = torch.tensor(lengths, dtype=torch.long)
    return input_ids, lengths_tensor, token_lists

def infer_texts(model, vocab, texts, max_len, device):
    input_ids, lengths, token_lists = preprocess_texts_for_inference(texts=texts, vocab=vocab, max_len=max_len)
    input_ids = input_ids.to(device)
    lengths = lengths.to(device)

    model.eval()
    with torch.no_grad():
        output = model(input_ids, lengths)
        if isinstance(output, tuple):
            probs, attention_weights, _ = output
        else:
            probs = output
            attention_weights = None

    probs = probs.squeeze(1).detach().cpu()
    if attention_weights is not None:
        attention_weights = attention_weights.detach().cpu()

    for idx, text in enumerate(texts):
        label = '긍정' if probs[idx].item() >= 0.5 else '부정'
        tokens = token_lists[idx]
        seq_len = min(len(tokens), max_len)

        print(f'[샘플 {idx}] 문장: {text}')
        print(f'  예측 확률: {probs[idx].item():.4f} / 예측 라벨: {label}')

        if attention_weights is not None and seq_len > 0:
            used_weights = attention_weights[idx][:seq_len]
            top_k = min(3, seq_len)
            top_vals, top_pos = torch.topk(used_weights, k=top_k)
            print('  상위 attention 토큰:')
            for rank in range(top_k):
                token = tokens[top_pos[rank].item()]
                weight = top_vals[rank].item()
                print(f'    - {token}: {weight:.4f}')


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [2]:
cfg = TrainConfig()
set_seed(cfg.seed)

raw_df = load_raw_dataframe(csv_path=None, csv_url=DEFAULT_URL)
train_df = build_train_dataframe(raw_df, threshold=cfg.rating_threshold)

processed_df, vocab = preprocess_dataframe(
    df=train_df,
    text_col='review',
    label_col='sentiment',
    max_len=cfg.max_len,
    stopwords=DEFAULT_STOPWORDS,
    min_freq=cfg.min_freq,
)
dataset = KoreanSentimentDataset(processed_df)

print('[raw data shape]', raw_df.shape)
print('[processed data shape]', processed_df.shape)
print('[dataset size]', len(dataset))
print('[vocab size]', len(vocab))
print('[min length]', int(dataset.lengths.min().item()))
print('[max length]', int(dataset.lengths.max().item()))
print(processed_df[['review', 'clean_text', 'tokens', 'padded_ids', 'length', 'label']].head())

[raw data shape] (14725, 4)
[processed data shape] (14725, 8)
[dataset size] 14725
[vocab size] 54967
[min length] 1
[max length] 30
                                              review  \
0                             돈 들인건 티가 나지만 보는 내내 하품만   
1       몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.   
2  이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...   
3                                이 정도면 볼만하다고 할 수 있음!   
4                                               재미있다   

                                          clean_text  \
0                             돈 들인건 티가 나지만 보는 내내 하품만   
1          몰입할수밖에 없다 어렵게 생각할 필요없다 내가 전투에 참여한듯 손에 땀이남   
2  이전 작품에 비해 더 화려하고 스케일도 커졌지만 전국 맛집의 음식들을 한데 모은 것...   
3                                 이 정도면 볼만하다고 할 수 있음   
4                                               재미있다   

                                              tokens  \
0                     [돈, 들인건, 티가, 나지만, 보는, 내내, 하품만]   
1  [몰입할수밖에, 없다, 어렵게, 생각할, 필요없다, 내가, 전투에, 참여한듯, 손에...   
2  [이전, 작품에, 비해, 화려하고, 스케

In [3]:
val_size = max(1, int(len(dataset) * cfg.val_ratio)) if len(dataset) > 1 else 0
train_size = len(dataset) - val_size

if val_size > 0:
    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(cfg.seed),
    )
else:
    train_dataset = dataset
    val_dataset = dataset

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False)

print('[train batches]', len(train_loader))
print('[val batches]', len(val_loader))

[train batches] 369
[val batches] 93


In [4]:
baseline_model = BiLSTMSentimentClassifier(
    vocab_size=len(vocab),
    embedding_dim=cfg.embedding_dim,
    hidden_dim=cfg.hidden_dim,
    num_layers=1,
    dropout=cfg.dropout,
    pad_idx=vocab['<pad>'],
    verbose=False,
).to(device)

attention_model = BiLSTMAttentionSentimentClassifier(
    vocab_size=len(vocab),
    embedding_dim=cfg.embedding_dim,
    hidden_dim=cfg.hidden_dim,
    attention_dim=cfg.attention_dim,
    dropout=cfg.dropout,
    pad_idx=vocab['<pad>'],
    verbose=False,
).to(device)

baseline_model

BiLSTMSentimentClassifier(
  (embedding): Embedding(54967, 64, padding_idx=0)
  (bilstm): LSTM(64, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [5]:
baseline_history = fit_model(
    model=baseline_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=cfg.epochs,
    lr=cfg.lr,
    device=device,
    name='BiLSTM',
)

attention_history = fit_model(
    model=attention_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=cfg.epochs,
    lr=cfg.lr,
    device=device,
    name='BiLSTM+Attention',
)

print('\n[성능 비교]')
print('model			train_loss	train_acc	val_loss	val_acc')
print(
    f"BiLSTM			{baseline_history[-1]['train_loss']:.4f}		{baseline_history[-1]['train_acc']:.4f}		{baseline_history[-1]['val_loss']:.4f}		{baseline_history[-1]['val_acc']:.4f}"
)
print(
    f"BiLSTM+Attention	{attention_history[-1]['train_loss']:.4f}		{attention_history[-1]['train_acc']:.4f}		{attention_history[-1]['val_loss']:.4f}		{attention_history[-1]['val_acc']:.4f}"
)


[BiLSTM] start training
[BiLSTM] epoch 1: train_loss=0.5488, train_acc=0.7560, val_loss=0.5105, val_acc=0.7705
[BiLSTM] epoch 2: train_loss=0.4265, train_acc=0.8063, val_loss=0.4660, val_acc=0.7920
[BiLSTM] epoch 3: train_loss=0.2771, train_acc=0.8834, val_loss=0.4759, val_acc=0.7967

[BiLSTM+Attention] start training
[BiLSTM+Attention] epoch 1: train_loss=0.5334, train_acc=0.7591, val_loss=0.4970, val_acc=0.7712
[BiLSTM+Attention] epoch 2: train_loss=0.4109, train_acc=0.8125, val_loss=0.4482, val_acc=0.7950
[BiLSTM+Attention] epoch 3: train_loss=0.2822, train_acc=0.8810, val_loss=0.4802, val_acc=0.7876

[성능 비교]
model			train_loss	train_acc	val_loss	val_acc
BiLSTM			0.2771		0.8834		0.4759		0.7967
BiLSTM+Attention	0.2822		0.8810		0.4802		0.7876


In [6]:
sample_texts = [
    '돈 들인건 티가 나지만 보는 내내 하품만',
    '몰입할수밖에 없다 어렵게 생각할 필요없다 내가 전투에 참여한듯 손에 땀이남',
]

print('\n[BiLSTM 추론]')
infer_texts(
    model=baseline_model,
    vocab=vocab,
    texts=sample_texts,
    max_len=cfg.max_len,
    device=device,
)

print('\n[BiLSTM+Attention 추론]')
infer_texts(
    model=attention_model,
    vocab=vocab,
    texts=sample_texts,
    max_len=cfg.max_len,
    device=device,
)


[BiLSTM 추론]
[샘플 0] 문장: 돈 들인건 티가 나지만 보는 내내 하품만
  예측 확률: 0.0551 / 예측 라벨: 부정
[샘플 1] 문장: 몰입할수밖에 없다 어렵게 생각할 필요없다 내가 전투에 참여한듯 손에 땀이남
  예측 확률: 0.9515 / 예측 라벨: 긍정

[BiLSTM+Attention 추론]
[샘플 0] 문장: 돈 들인건 티가 나지만 보는 내내 하품만
  예측 확률: 0.0763 / 예측 라벨: 부정
  상위 attention 토큰:
    - 돈: 0.9184
    - 들인건: 0.0393
    - 나지만: 0.0211
[샘플 1] 문장: 몰입할수밖에 없다 어렵게 생각할 필요없다 내가 전투에 참여한듯 손에 땀이남
  예측 확률: 0.9558 / 예측 라벨: 긍정
  상위 attention 토큰:
    - 손에: 0.2957
    - 몰입할수밖에: 0.1444
    - 필요없다: 0.1347
